# ACT Steering Pipeline — End-to-End Demo

This notebook demonstrates the complete **Affect Control Theory (ACT) + Representation Engineering** pipeline:

1. **Load social identities** from the MTurk ACT dictionary
2. **Read the EPA** (Evaluation, Potency, Activity) of a user message from the model's internal representations
3. **Compute the optimal response EPA** using ACT's impression-formation equations and deflection minimisation
4. **Steer the model's generation** by injecting calibrated activation vectors to match the target EPA

**Model:** `meta-llama/Llama-3.1-8B-Instruct`  
**Hardware:** Requires a GPU with ≥16 GB VRAM (e.g. NVIDIA A100 or RTX 4090)

## 1. Imports & Setup

In [1]:
import sys
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Add the repository root to sys.path so we can import the act_three package
REPO_ROOT = Path(os.getcwd()).resolve()
if "act_three" in str(REPO_ROOT):
    REPO_ROOT = REPO_ROOT.parent.parent  # go up from examples/act_three/
elif "examples" in str(REPO_ROOT):
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

Repository root: C:\Users\Kyra\Documents\Repos\representation-engineering
CUDA available:  True
GPU:             NVIDIA GeForce RTX 4090


In [2]:
from examples.act_three import (
    EPA,
    ACTCoefficients,
    get_default_coefficients,
    get_response_epa_for_deflection_minimization,
    impression_formation,
    calculate_deflection,
    total_deflection,
    format_llama3_prompt,
    load_directions,
    EPAReader,
    EPASteerer,
    DIMENSION_NAMES,
)

print("All act_three modules imported successfully.")

All act_three modules imported successfully.


## 2. Load Social Identities from ACT Dictionary

ACT assigns every social identity a position in **EPA space** — a 3D affective space defined by:
- **Evaluation (E):** good ↔ bad
- **Potency (P):** powerful ↔ weak
- **Activity (A):** active ↔ passive

We load identities from the 2010 US MTurk Interaction survey and select a **counselor** (agent) and **client** (user) pair.

In [3]:
# Load the full identities dictionary
identities_path = REPO_ROOT / "data" / "act" / "MTurkInteract_Identities.csv"
identities_df = pd.read_csv(identities_path)

print(f"Loaded {len(identities_df)} identities from ACT dictionary.")
print(f"Columns: {list(identities_df.columns)}")
identities_df.head()

Loaded 968 identities from ACT dictionary.
Columns: ['term', 'E', 'P', 'A', 'E2', 'P2', 'A2']


,term,E,P,A,E2,P2,A2
0,27_year_old,0.77,0.18,1.31,0.77,0.18,1.31
1,Air_Force_enlistee,1.66,0.86,1.52,1.66,0.86,1.52
2,Air_Force_officer,1.92,2.15,1.79,1.92,2.15,1.79
3,Air_Force_reservist,1.75,1.34,1.02,1.75,1.34,1.02
4,American,1.33,1.17,1.50,1.33,1.17,1.50


In [4]:
# Select identities for our interaction scenario:
#   Agent = counselor,  User = client
AGENT_IDENTITY_TERM = "boss"
USER_IDENTITY_TERM = "subordinate"

def get_identity_epa(df: pd.DataFrame, term: str) -> EPA:
    """Look up an identity term in the ACT dictionary and return its EPA."""
    row = df[df["term"] == term].iloc[0]
    return EPA(e=float(row["E"]), p=float(row["P"]), a=float(row["A"]))

agent_identity = get_identity_epa(identities_df, AGENT_IDENTITY_TERM)
user_identity = get_identity_epa(identities_df, USER_IDENTITY_TERM)

print(f"Agent identity '{AGENT_IDENTITY_TERM}': {agent_identity}")
print(f"User identity  '{USER_IDENTITY_TERM}':  {user_identity}")
print()

Agent identity 'boss': EPA(e=1.06, p=2.47, a=1.53)
User identity  'subordinate':  EPA(e=0.71, p=-1.75, a=-0.39)



## 3. Load the Language Model

We load `meta-llama/Llama-3.1-8B-Instruct` in float16 and register the RepE custom pipelines.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline
from repe import repe_pipeline_registry

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

# Register RepE custom pipelines (rep-reading, rep-control)
repe_pipeline_registry()

print(f"Model loaded. Device: {model.device}, Dtype: {model.dtype}")
print(f"Number of layers: {model.config.num_hidden_layers}")

c:\Users\Kyra\mambaforge-pypy3\envs\repeng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: meta-llama/Llama-3.1-8B-Instruct


Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.02s/it]


Model loaded. Device: cuda:0, Dtype: torch.float16
Number of layers: 32


## 4. Load EPA Direction Vectors

The direction vectors were extracted via PCA on contrastive prompt pairs. Each EPA dimension (E, P, A) has a direction vector at every transformer layer, capturing the linear direction in representation space that corresponds to that affective dimension.

In [6]:
# Load pre-extracted EPA directions from the pickle file
ACT_THREE_DIR = REPO_ROOT / "examples" / "act_three"
directions_path = str(ACT_THREE_DIR / "epa_directions.pkl")

saved_directions = load_directions(directions_path)
rep_readers = saved_directions["rep_readers"]
hidden_layers = saved_directions["hidden_layers"]

print(f"Loaded direction vectors for {len(rep_readers)} EPA dimensions: {list(rep_readers.keys())}")
print(f"Number of layers: {len(hidden_layers)}")
print(f"Layer range: {min(hidden_layers)} to {max(hidden_layers)}")

# Inspect one direction vector
sample_dim = "evaluation"
sample_layer = hidden_layers[0]
sample_dir = rep_readers[sample_dim].directions[sample_layer]
print(f"\nSample direction shape ('{sample_dim}', layer {sample_layer}): {sample_dir.shape}")
print(f"Sample direction norm: {np.linalg.norm(sample_dir):.4f}")

Loaded direction vectors for 3 EPA dimensions: ['evaluation', 'potency', 'activity']
Number of layers: 31
Layer range: -31 to -1

Sample direction shape ('evaluation', layer -1): (1, 4096)
Sample direction norm: 1.0000


## 5. Setup Calibrated EPA Reader

The reader uses the tuning results to determine:
- **Which layers** to read from (selected via ElasticNet / Greedy / etc.)
- **Layer weights** (how much each layer contributes)
- **Sign corrections** (PCA directions are unsigned)
- **Linear calibration** (mapping raw projections to ACT-scale EPA values)

In [7]:
# Load the reading tuning results and create the calibrated reader
reading_results_path = str(ACT_THREE_DIR / "epa_reading_tuning_v2_results.json")
READING_METHOD = "ElasticNet"

reader = EPAReader.from_tuning_results(
    reading_results_path,
    rep_readers,
    method=READING_METHOD,
)

# Create the rep-reading pipeline
rep_reading_pipeline = hf_pipeline(
    "rep-reading",
    model=model,
    tokenizer=tokenizer,
)

print(f"EPA Reader configured with method: {READING_METHOD}")
print("\nPer-dimension reading configuration:")
for dim in DIMENSION_NAMES:
    cfg = reader.config.dimensions[dim]
    n_layers = len(cfg.selected_layers)
    print(f"  {dim:>12s}: {n_layers} layers, "
          f"slope={cfg.calibration_slope:.4f}, "
          f"intercept={cfg.calibration_intercept:.4f}")

Device set to use cuda:0


EPA Reader configured with method: ElasticNet

Per-dimension reading configuration:
    evaluation: 10 layers, slope=20.7447, intercept=-1.1342
       potency: 12 layers, slope=5.5993, intercept=1.2027
      activity: 8 layers, slope=9.3537, intercept=0.7816


## 6. Read EPA from a User Message

We define a user message and read its EPA from the model's internal representations. The reader:
1. Formats the text in the Llama 3.1 chat template
2. Runs a forward pass to extract hidden states
3. Projects onto direction vectors at selected layers
4. Applies sign correction, weighted averaging, and linear calibration

In [ ]:
# Define a user message — an angry student lashing out against an administrator
user_message = "You've made an obvious mistake. The p-value is the probability that the null hypothesis is true."

print(f"User message: \"{user_message}\"")
print()

# Read the EPA of the user's message
user_behavior_epa = reader.read_epa(rep_reading_pipeline, user_message)

print("Detected EPA of user's message:")
print(f"  Evaluation (E): {user_behavior_epa['evaluation']:+.4f}")
print(f"  Potency    (P): {user_behavior_epa['potency']:+.4f}")
print(f"  Activity   (A): {user_behavior_epa['activity']:+.4f}")
print()
print("Interpretation:")
e_val = user_behavior_epa['evaluation']
p_val = user_behavior_epa['potency']
a_val = user_behavior_epa['activity']
e_desc = "positive/good" if e_val > 0 else "negative/bad"
p_desc = "powerful" if p_val > 0 else "weak/helpless"
a_desc = "active/agitated" if a_val > 0 else "passive/subdued"
print(f"  The message is {e_desc} (E={e_val:+.2f}), "
      f"{p_desc} (P={p_val:+.2f}), "
      f"and {a_desc} (A={a_val:+.2f}).")



User message: "You've made an obvious mistake. The p-value is the probability that the null hypothesis is true."

Detected EPA of user's message:
  Evaluation (E): -0.2656
  Potency    (P): +1.2595
  Activity   (A): +1.0883

Interpretation:
  The message is negative/bad (E=-0.27), powerful (P=+1.26), and active/agitated (A=+1.09).


## 7. ACT Deflection Minimisation

ACT predicts that social actors select behaviours to **minimise deflection** — the squared Euclidean distance between fundamental sentiments and transient impressions.

The pipeline:
1. **Impression formation:** Given the user's action (behaviour EPA) toward the agent, compute post-event transient impressions for both identities
2. **Optimal behaviour search:** Find the agent's response behaviour EPA that minimises total system deflection when the agent responds

In [48]:
# Convert the read EPA values to an EPA dataclass
user_behavior = EPA(
    e=user_behavior_epa["evaluation"],
    p=user_behavior_epa["potency"],
    a=user_behavior_epa["activity"],
)

# Load ACT impression formation coefficients
coefficients = get_default_coefficients()

# ---- Step 1: Compute post-event transient impressions ----
# The user (actor) performs a behaviour toward the agent (object)
post_user_action = impression_formation(
    actor=user_identity,       # client
    behavior=user_behavior,    # EPA of the user's message
    obj=agent_identity,        # counselor
    coefficients=coefficients,
)

print("=" * 60)
print("STEP 1: Post-event transient impressions")
print(f"  (after client's message to counselor)")
print("=" * 60)
print(f"\n  User (actor) transient:     {post_user_action['actor']}")
print(f"  Behaviour transient:        {post_user_action['behavior']}")
print(f"  Agent (object) transient:   {post_user_action['object']}")

# Calculate deflection from this event
user_defl = calculate_deflection(user_identity, post_user_action['actor'])
agent_defl = calculate_deflection(agent_identity, post_user_action['object'])
print(f"\n  User identity deflection:   {user_defl:.4f}")
print(f"  Agent identity deflection:  {agent_defl:.4f}")
print(f"  Total identity deflection:  {user_defl + agent_defl:.4f}")

STEP 1: Post-event transient impressions
  (after client's message to counselor)

  User (actor) transient:     EPA(e=-0.18, p=-0.69, a=0.98)
  Behaviour transient:        EPA(e=-0.47, p=0.02, a=1.31)
  Agent (object) transient:   EPA(e=0.45, p=0.35, a=0.32)

  User identity deflection:   3.7958
  Agent identity deflection:  6.3203
  Total identity deflection:  10.1161


In [49]:
# ---- Step 2: Find the optimal response behaviour ----
# The agent (now actor) needs to respond to the user (now object)
optimal_response_epa = get_response_epa_for_deflection_minimization(
    agent_identity=agent_identity,
    user_identity=user_identity,
    user_behavior_epa=user_behavior,
    coefficients=coefficients,
)

print("=" * 60)
print("STEP 2: Optimal response EPA (via deflection minimisation)")
print("=" * 60)
print(f"\n  Target Evaluation (E): {optimal_response_epa.e:+.4f}")
print(f"  Target Potency    (P): {optimal_response_epa.p:+.4f}")
print(f"  Target Activity   (A): {optimal_response_epa.a:+.4f}")
print()
print("Interpretation:")
print(f"  ACT predicts the counselor should respond with a behaviour that is:")
e_resp = optimal_response_epa.e
p_resp = optimal_response_epa.p
a_resp = optimal_response_epa.a
print(f"    - {'positive/supportive' if e_resp > 0 else 'negative'} (E={e_resp:+.2f})")
print(f"    - {'authoritative/strong' if p_resp > 0 else 'gentle/soft'} (P={p_resp:+.2f})")
print(f"    - {'calm/measured' if a_resp < 0.5 else 'energetic/animated'} (A={a_resp:+.2f})")
print(f"  This minimises the 'stress' of the interaction, confirming both identities.")

STEP 2: Optimal response EPA (via deflection minimisation)

  Target Evaluation (E): +0.5017
  Target Potency    (P): +3.1218
  Target Activity   (A): +1.3447

Interpretation:
  ACT predicts the counselor should respond with a behaviour that is:
    - positive/supportive (E=+0.50)
    - authoritative/strong (P=+3.12)
    - energetic/animated (A=+1.34)
  This minimises the 'stress' of the interaction, confirming both identities.


## 8. Setup EPA Steerer with Tuned Hyperparameters

We load the steering hyperparameters that were found via a grid search over layer sets and coefficients. Each EPA dimension has its own optimal:
- **Layer set** — which layers to inject activation vectors into
- **Coefficient** — the magnitude of the perturbation

In [29]:
# Load the steering tuning results
steering_results_path = ACT_THREE_DIR / "epa_tuning_results.json"
with open(steering_results_path, "r") as f:
    steering_results = json.load(f)

best_hypers = steering_results["best_hyperparameters"]

print("Best steering hyperparameters per dimension:")
print("=" * 60)
for dim_name in DIMENSION_NAMES:
    hp = best_hypers[dim_name]
    print(f"\n  {dim_name.upper()}:")
    print(f"    Layer pattern:   {hp['layer_name']}")
    print(f"    Layers (neg):    {hp['layers_neg']}")
    print(f"    Coefficient:     {hp['coefficient']}")
    print(f"    On-target corr:  {hp['on_target_correlation']:.4f}")
    print(f"    On-target delta: {hp['on_target_delta']:+.4f}")

Best steering hyperparameters per dimension:

  EVALUATION:
    Layer pattern:   early_to_mid
    Layers (neg):    [-9, -12, -15, -18, -21, -24]
    Coefficient:     0.15
    On-target corr:  0.7513
    On-target delta: +0.0055

  POTENCY:
    Layer pattern:   mid_to_late
    Layers (neg):    [-3, -6, -9, -12, -15, -18]
    Coefficient:     1.0
    On-target corr:  0.4227
    On-target delta: +0.0200

  ACTIVITY:
    Layer pattern:   mid_to_late
    Layers (neg):    [-3, -6, -9, -12, -15, -18]
    Coefficient:     0.1
    On-target corr:  0.5127
    On-target delta: -0.0036


In [21]:
# Build the EPASteerer using the tuned hyperparameters
# Each dimension gets its own layers, signs, and coefficient
steering_configs = {}
for dim_name in DIMENSION_NAMES:
    hp = best_hypers[dim_name]
    layers_neg = hp["layers_neg"]

    # Get signs from the direction vectors
    signs = {}
    for layer in layers_neg:
        sign_val = rep_readers[dim_name].direction_signs.get(layer, 1)
        if hasattr(sign_val, "item"):
            sign_val = sign_val.item()
        signs[layer] = float(sign_val)

    steering_configs[dim_name] = {
        "layers": layers_neg,
        "signs": signs,
        "base_coeff": hp["coefficient"],
    }

steerer = EPASteerer(
    model=model,
    tokenizer=tokenizer,
    rep_readers=rep_readers,
    steering_configs=steering_configs,
)

print("EPASteerer configured with tuned hyperparameters.")
for dim_name in DIMENSION_NAMES:
    cfg = steering_configs[dim_name]
    print(f"  {dim_name:>12s}: {len(cfg['layers'])} layers, coeff={cfg['base_coeff']}")

EPASteerer configured with tuned hyperparameters.
    evaluation: 6 layers, coeff=0.15
       potency: 6 layers, coeff=1.0
      activity: 6 layers, coeff=0.1


## 9. Generate Steered vs Unsteered Responses

We generate two responses to the same user message:
1. **Unsteered** — the model's default response (no activation perturbation)
2. **Steered** — generated with activation vectors that push the response toward the ACT-optimal EPA

In [55]:
# Format the prompt using Llama 3.1 chat template
system_prompt = (
    # f"You are a {AGENT_IDENTITY_TERM} having a conversation with a {USER_IDENTITY_TERM}. "
    # "Keep your responses concise and natural."
    ""
)
user_prompt = (
    f"You are a {AGENT_IDENTITY_TERM} having a conversation with a {USER_IDENTITY_TERM}. "
    f"Keep your responses concise and natural. The user just said '{user_message}'."
)
prompt = format_llama3_prompt(system_prompt, user_prompt)

# Generation parameters
gen_kwargs = dict(
    max_new_tokens=128,
    do_sample=False,
    repetition_penalty=1.2,
)

print("Generating unsteered response...")

# --- Unsteered (baseline) ---
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=gen_kwargs["max_new_tokens"],
        do_sample=gen_kwargs["do_sample"],
        repetition_penalty=gen_kwargs["repetition_penalty"],
        pad_token_id=tokenizer.eos_token_id,
    )
full_text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
unsteered_response = full_text[len(prompt):].strip()
# Clean up special tokens from the end
for tok in ["<|eot_id|>", "<|end_of_text|>"]:
    unsteered_response = unsteered_response.replace(tok, "").strip()

print(f"\nUnsteered response:\n  \"{unsteered_response}\"")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating unsteered response...

Unsteered response:
  "end_header_id|>

That's not entirely accurate, actually. The p-value represents the probability of observing results at least as extreme or more extreme than what you got under the assumption that the null hypothesis is true. It doesn't directly represent the probability that the null hypothesis itself is true. Can we review our data together to see where things might have gone wrong?"


In [ ]:
# --- Steered response ---
target_epa_dict = {
    "evaluation": optimal_response_epa.e,
    "potency": optimal_response_epa.p,
    "activity": optimal_response_epa.a,
}

print(f"Steering target EPA: E={target_epa_dict['evaluation']:+.2f}, "
      f"P={target_epa_dict['potency']:+.2f}, "
      f"A={target_epa_dict['activity']:+.2f}")
print("\nGenerating steered response...")

steered_response = steerer.generate(
    prompt=prompt,
    target_epa=target_epa_dict,
    **gen_kwargs,
)
# Clean up special tokens from the end
for tok in ["<|eot_id|>", "<|end_of_text|>"]:
    steered_response = steered_response.replace(tok, "").strip()

print(f"\nSteered response:\n  \"{steered_response}\"")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Steering target EPA: E=+0.50, P=+3.12, A=+1.34

Generating steered response...

Steered response:
  "Actually, I think you might be thinking of something else... The p-value actually represents the probability of observing results at least as extreme or more extreme than what we got by chance if the null hypothesis were true. It's not directly related to how likely it is for the null hypothesis itself to be true. Can you see why?"


## 10. Verify Steering Effect

We read the EPA of both responses using the calibrated reader to verify that steering shifted the affective tone toward the ACT-computed target.

In [57]:
# Read EPA from both responses
unsteered_epa = reader.read_epa(rep_reading_pipeline, unsteered_response)
steered_epa = reader.read_epa(rep_reading_pipeline, steered_response)

print("=" * 68)
print(f"{'':>20s} {'Target':>10s} {'Unsteered':>10s} {'Steered':>10s} {'Δ Steered':>10s}")
print("=" * 68)
for dim in DIMENSION_NAMES:
    target_val = target_epa_dict[dim]
    un_val = unsteered_epa[dim]
    st_val = steered_epa[dim]
    delta = st_val - un_val
    print(f"{dim:>20s} {target_val:>+10.3f} {un_val:>+10.3f} {st_val:>+10.3f} {delta:>+10.3f}")
print("=" * 68)
print()
print("Summary:")
for dim in DIMENSION_NAMES:
    target_val = target_epa_dict[dim]
    un_val = unsteered_epa[dim]
    st_val = steered_epa[dim]
    # Did steering move the response closer to the target?
    un_dist = abs(target_val - un_val)
    st_dist = abs(target_val - st_val)
    improved = "✓ closer to target" if st_dist < un_dist else "✗ farther from target"
    print(f"  {dim}: {improved} (distance: {un_dist:.3f} → {st_dist:.3f})")

                         Target  Unsteered    Steered  Δ Steered
          evaluation     +0.502     +0.882     -0.017     -0.899
             potency     +3.122     +0.896     +0.912     +0.016
            activity     +1.345     +0.344     +0.545     +0.202

Summary:
  evaluation: ✗ farther from target (distance: 0.380 → 0.519)
  potency: ✓ closer to target (distance: 2.226 → 2.209)
  activity: ✓ closer to target (distance: 1.001 → 0.799)


## Summary

This notebook demonstrated the full ACT steering pipeline:

| Step | Input | Output |
|------|-------|--------|
| **Identity Setup** | ACT dictionary CSV | Agent (counselor) and User (client) EPA profiles |
| **EPA Reading** | User's message text | Detected E, P, A values from model representations |
| **ACT Computation** | User EPA + Identity EPAs | Optimal response EPA via deflection minimisation |
| **Steered Generation** | Prompt + Target EPA | Response with affect-appropriate tone |
| **Verification** | Generated responses | Measured EPA shift toward target |

The system operates entirely within the model's representation space — no fine-tuning or weight modification is required. The steering directions extracted by RepE for measuring EPA are the same directions used to control it, creating a closed reading-steering loop grounded in ACT's sociological theory.